Training_Pairs -- deki 1'ler Ground Truth yani direkt müşterinin gidip tıklamasıyla oluşmuş büüyk ihitmalle.Biz ise iki tanenin ilişkisine bakarken bunlar %78 ihtimalle alakalı,, eşik değeri de 0.5 se bu bir 1(relevant) etikettir dicez.

In [2]:
!pip install thefuzz lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import random
import pandas as pd
import numpy as np
import lightgbm as lgb
from thefuzz import fuzz
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pandas as pd

items = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/items.csv')
sample_submission = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/sample_submission.csv')
submission_pairs = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/submission_pairs.csv')
terms = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/terms.csv')
training_pairs = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/training_pairs.csv')

In [5]:
items.head()

,item_id,title,category,brand,gender,age_group,attributes
0,ITEM_3a515fb7125d,erkek kumaş usb kulaklık çıkışlı bodybag göğüs...,aksesuar/çanta/omuz çantası,newish polo,erkek,yetişkin,"materyal: tekstil, deri kalitesi: parça mevcut..."
1,ITEM_94fe5db929cd,orijinal italyan charm bileklik kişiselleştiri...,aksesuar/takı & mücevher/bileklik/çelik bileklik,nomination,unisex,yetişkin,"taş cinsi: yok, renk: gri, materyal: paslanmaz..."
2,ITEM_576afa5db9a1,kupa bardak alchemist high,ev & mobilya/ev/sofra & mutfak/sofra/bardak,qivi,unknown,unknown,"materyal: seramik, parça sayısı: 1, renk: beya..."
3,ITEM_846531067083,çizgili geniş kol koton gömlek,giyim/üst giyim/gömlek,ayhan,kadın,yetişkin,"desen: çizgili, kalıp: regular, yaka tipi: v y..."
4,ITEM_1c3943c4d220,kapaklı katlanabilir puantiyeli pembe 60 litre...,ev & mobilya/ev/ev gereçleri/çamaşırlık ev ger...,serstil,unknown,unknown,"materyal: polipropilen, renk: pembe, hacim: 60..."


In [6]:
items.nunique()

,0
item_id,962873
title,915525
category,2932
brand,79790
gender,4
age_group,6
attributes,730697


In [7]:
print('--- Category Information ---')
display(pd.Series(items['category'].unique(), name='Unique Categories'))
print(f'Number of unique categories: {items["category"].nunique()}')

print('\n--- Age Group Information ---')
display(pd.Series(items['age_group'].unique(), name='Unique Age Groups'))
print(f'Number of unique age groups: {items["age_group"].nunique()}')

print('\n--- Gender Group Information ---')
display(pd.Series(items['gender'].unique(), name='Unique Age Groups'))
print(f'Number of unique age groups: {items["gender"].nunique()}')

--- Category Information ---


,Unique Categories
0,aksesuar/çanta/omuz çantası
1,aksesuar/takı & mücevher/bileklik/çelik bileklik
2,ev & mobilya/ev/sofra & mutfak/sofra/bardak
3,giyim/üst giyim/gömlek
4,ev & mobilya/ev/ev gereçleri/çamaşırlık ev ger...
...,...
2927,otomobil & motosiklet/motosiklet/motosiklet ye...
2928,spor & outdoor/ekipman & aksesuar/basketbol/ba...
2929,spor & outdoor/ekipman & aksesuar/okçuluk spor...
2930,bahçe & elektrikli el aletleri/bahçe/depolama ...


Number of unique categories: 2932

--- Age Group Information ---


,Unique Age Groups
0,yetişkin
1,unknown
2,çocuk
3,genç
4,bebek
5,bebek & çocuk


Number of unique age groups: 6

--- Gender Group Information ---


,Unique Age Groups
0,erkek
1,unisex
2,unknown
3,kadın


Number of unique age groups: 4


In [8]:
terms.head()

,term_id,query
0,TERM_f2b61db2,defacto kız bebek elbise
1,TERM_00247623,bebek oto koltuğu
2,TERM_c977024b,sava lastik
3,TERM_dbdbddbb,kadın tesettür kışlık gömlek
4,TERM_c06329e3,erkek çocuk kranpon


In [9]:
terms.nunique()

,0
term_id,50153
query,50153


In [10]:
training_pairs.head()

,id,term_id,item_id,label
0,TRN_c639ed31a5,TERM_68c2d117,ITEM_0c59058c3908,1
1,TRN_ca3ce092f7,TERM_3374e63e,ITEM_d0d66d79750c,1
2,TRN_d4e3637fd6,TERM_a2d68022,ITEM_cd5152c2807c,1
3,TRN_eaaa1cecbb,TERM_161ac15a,ITEM_f1ebadc41d01,1
4,TRN_42ef8efcad,TERM_6331588a,ITEM_e7a0765b4788,1


In [11]:
print('DataFrame shape (rows, columns):', training_pairs.shape)

DataFrame shape (rows, columns): (250000, 4)


In [12]:
submission_pairs.head()

,id,term_id,item_id
0,TST_8ede5e2b0176ec,TERM_194bea06,ITEM_41f4c31b7c05
1,TST_2a09d773d97e6e,TERM_eacb5395,ITEM_974f6e875b57
2,TST_a4a86755c99d3d,TERM_07ab54b2,ITEM_d81d95b30383
3,TST_e2db221eaae0ec,TERM_80f9a29f,ITEM_027ce477fcac
4,TST_2718f17b96c6f5,TERM_2b89cd08,ITEM_b0897b7143fa


In [13]:
submission_pairs.shape

(3359679, 3)

In [14]:
print("1. Veriler yükleniyor...")
items = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/items.csv')
terms = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/terms.csv')
training_pairs = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/training_pairs.csv')
submission_pairs = pd.read_csv('/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/submission_pairs.csv')

# Orijinal pozitif verileri işaretliyoruz
training_pairs['label'] = 1

print("2. Test setinden (submission_pairs) 1 Milyonluk havuz oluşturuluyor...")
pool = submission_pairs.sample(n=1000000, random_state=42).copy()

# Arama terimi ve ürün metinlerini (title, category vb.) yanlarına getiriyoruz
pool = pool.merge(terms, on='term_id', how='left')
pool = pool.merge(items, on='item_id', how='left')

print("3. Metinler standartlaştırılıyor...")
pool['query'] = pool['query'].astype(str).str.lower()
pool['title'] = pool['title'].astype(str).str.lower()
pool['gender'] = pool['gender'].astype(str).str.lower()
pool['category'] = pool['category'].astype(str).str.lower()

print("4. Cinsiyet zıtlığı filtresi uygulanıyor (Hızlı İşlem)...")
mask_gender_mismatch = (
    (pool['query'].str.contains('erkek') & (pool['gender'] == 'kadın')) |
    (pool['query'].str.contains('kadın') & (pool['gender'] == 'erkek'))
)

print("5. Kelime kesişimi filtresi uygulanıyor (Yaklaşık 2-4 dakika sürebilir, bekleyiniz)...")
def is_totally_unrelated(row):
    query_words = set(row['query'].split())
    target_text = row['title'] + " " + row['category'].replace('/', ' ')
    target_words = set(target_text.split())
    return len(query_words.intersection(target_words)) == 0

pool['is_unrelated'] = pool.apply(is_totally_unrelated, axis=1)

# Altın değerindeki kesin negatiflerimizi süzüyoruz
hard_negatives = pool[mask_gender_mismatch | pool['is_unrelated']].copy()
print(f"Filtreleme sonucu bulunan Kesin Negatif sayısı: {len(hard_negatives)}")

print("6. Eğitim setine eklenecek 250.000 adet negatif seçiliyor...")
TARGET_NEGATIVE_COUNT = 250000
if len(hard_negatives) > TARGET_NEGATIVE_COUNT:
    final_negatives = hard_negatives.sample(n=TARGET_NEGATIVE_COUNT, random_state=42)
else:
    print("Not: Havuzdan hedeflenenden az negatif çıktı, hepsi alınıyor.")
    final_negatives = hard_negatives

# Çıktıyı modelin istediği eğitim formatına uyduruyoruz
final_negatives = final_negatives[['id', 'term_id', 'item_id']].copy()
final_negatives['label'] = 0

print("7. Pozitif ve Negatif veriler birleştirilip karıştırılıyor...")
# İŞTE BURADA TRAIN_READY TABLOSUNU YARATIYORUZ
train_ready = pd.concat([training_pairs[['id', 'term_id', 'item_id', 'label']], final_negatives], ignore_index=True)
train_ready = train_ready.sample(frac=1, random_state=42).reset_index(drop=True)

print("--------------------------------------------------")
print("✅ 1. HÜCRE İŞLEMİ BAŞARILI! Yeni Eğitim Setinin Sınıf Dağılımı:")
print(train_ready['label'].value_counts())
print("--------------------------------------------------")

1. Veriler yükleniyor...
2. Test setinden (submission_pairs) 1 Milyonluk havuz oluşturuluyor...
3. Metinler standartlaştırılıyor...
4. Cinsiyet zıtlığı filtresi uygulanıyor (Hızlı İşlem)...
5. Kelime kesişimi filtresi uygulanıyor (Yaklaşık 2-4 dakika sürebilir, bekleyiniz)...
Filtreleme sonucu bulunan Kesin Negatif sayısı: 575608
6. Eğitim setine eklenecek 250.000 adet negatif seçiliyor...
7. Pozitif ve Negatif veriler birleştirilip karıştırılıyor...
--------------------------------------------------
✅ 1. HÜCRE İŞLEMİ BAŞARILI! Yeni Eğitim Setinin Sınıf Dağılımı:
label
1    250000
0    250000
Name: count, dtype: int64
--------------------------------------------------


In [15]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from thefuzz import fuzz
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

print("1. Colab hafızasındaki 'train_ready' tablosuna metinler ekleniyor...")
df = train_ready.merge(terms, on='term_id', how='left')
df = df.merge(items, on='item_id', how='left')

cols_to_clean = ['query', 'title', 'brand', 'category', 'gender', 'age_group']
for col in cols_to_clean:
    df[col] = df[col].astype(str).str.lower().replace('nan', 'unknown')

print("2. Özellik Mühendisliği (Feature Engineering) Başladı (Ortalama 5-10 Dk Sürebilir)...")

# --- SENİN FİKRİN: ANA KATEGORİ ÇIKARIMI ---
df['ana_kategori'] = df['category'].apply(lambda x: x.split('/')[0]).astype('category')

def jaccard_sim(q, t):
    set_q, set_t = set(q.split()), set(t.split())
    if not set_q or not set_t: return 0.0
    return len(set_q.intersection(set_t)) / len(set_q.union(set_t))

def word_overlap(q, text):
    if text == 'unknown' or text == '': return 0
    return 1 if text in q else 0

print("   - Matematiksel benzerlikler hesaplanıyor...")
df['jaccard_sim'] = df.apply(lambda x: jaccard_sim(x['query'], x['title']), axis=1)
df['fuzz_ratio'] = df.apply(lambda x: fuzz.token_set_ratio(x['query'], x['title']) / 100.0, axis=1)
df['is_brand_in_query'] = df.apply(lambda x: word_overlap(x['query'], x['brand']), axis=1)
df['is_gender_in_query'] = df.apply(lambda x: word_overlap(x['query'], x['gender']), axis=1)
df['is_age_in_query'] = df.apply(lambda x: word_overlap(x['query'], x['age_group']), axis=1)
df['category_overlap'] = df.apply(lambda x: jaccard_sim(x['query'], x['category'].replace('/', ' ')), axis=1)

print("3. LightGBM Modeli Eğitiliyor (5-Fold Cross Validation)...")
features = [
    'jaccard_sim', 'fuzz_ratio', 'is_brand_in_query',
    'is_gender_in_query', 'is_age_in_query', 'category_overlap',
    'ana_kategori'
]

X = df[features]
y = df['label']
groups = df['term_id']

oof_preds = np.zeros(len(df))
trained_models = []
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=64,
        random_state=42,
        class_weight='balanced'
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=['ana_kategori'],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    trained_models.append(model)
    print(f"Fold {fold+1} tamamlandı. En iyi iterasyon: {model.best_iteration_}")

print("\n4. Macro F1 İçin Optimum Eşik Değeri Aranıyor...")
best_thresh = 0.5
best_f1 = 0

for thresh in np.arange(0.3, 0.7, 0.01):
    preds = (oof_preds > thresh).astype(int)
    score = f1_score(y, preds, average='macro')
    if score > best_f1:
        best_f1 = score
        best_thresh = thresh

print("--------------------------------------------------")
print(f"🎯 EĞİTİM SONUCU:")
print(f"En İyi Eşik Değeri (Threshold): {best_thresh:.2f}")
print(f"Beklenen Validation Macro F1 Skoru: {best_f1:.5f}")
print("--------------------------------------------------")

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': trained_models[0].feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nÖzellik Önem Sıralaması (Feature Importance):")
print(importance_df)

1. Colab hafızasındaki 'train_ready' tablosuna metinler ekleniyor...
2. Özellik Mühendisliği (Feature Engineering) Başladı (Ortalama 5-10 Dk Sürebilir)...
   - Matematiksel benzerlikler hesaplanıyor...
3. LightGBM Modeli Eğitiliyor (5-Fold Cross Validation)...
[LightGBM] [Info] Number of positive: 200773, number of negative: 199227
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004645 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 230
[LightGBM] [Info] Number of data points in the train set: 400000, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Fold 1 tamamlandı. En iyi iterasyon: 166
[LightGBM] [Info] Number of positive: 199839, number of negative: 200161
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the over

In [16]:
import pandas as pd
import numpy as np
from thefuzz import fuzz

print("1. Sınav Kağıdı (Test Verisi) Hazırlanıyor...")
# 3.36 milyonluk test setini (submission_pairs) alıyoruz
test_df = submission_pairs.copy()

# Arama terimi ve ürün metinlerini (title, category vb.) test setinin yanına getiriyoruz
test_df = test_df.merge(terms, on='term_id', how='left')
test_df = test_df.merge(items, on='item_id', how='left')

# Eksik verileri temizleme ve küçük harfe çevirme (Eğitimdeki gibi)
cols_to_clean = ['query', 'title', 'brand', 'category', 'gender', 'age_group']
for col in cols_to_clean:
    test_df[col] = test_df[col].astype(str).str.lower().replace('nan', 'unknown')

print("2. Özellik Mühendisliği Test Setine Uygulanıyor (Bu işlem biraz zaman alacak!)...")
# SENİN FİKRİN: Ana kategori çıkarımı
test_df['ana_kategori'] = test_df['category'].apply(lambda x: x.split('/')[0]).astype('category')

# Matematiksel sinyallerin (Jaccard, Fuzz vb.) 3.36 milyon satır için hesaplanması
print("   - Matematiksel benzerlikler hesaplanıyor (Lütfen kahvenizi alın, bu işlem 15-20 dakika sürebilir)...")
test_df['jaccard_sim'] = test_df.apply(lambda x: jaccard_sim(x['query'], x['title']), axis=1)
test_df['fuzz_ratio'] = test_df.apply(lambda x: fuzz.token_set_ratio(x['query'], x['title']) / 100.0, axis=1)
test_df['is_brand_in_query'] = test_df.apply(lambda x: word_overlap(x['query'], x['brand']), axis=1)
test_df['is_gender_in_query'] = test_df.apply(lambda x: word_overlap(x['query'], x['gender']), axis=1)
test_df['is_age_in_query'] = test_df.apply(lambda x: word_overlap(x['query'], x['age_group']), axis=1)
test_df['category_overlap'] = test_df.apply(lambda x: jaccard_sim(x['query'], x['category'].replace('/', ' ')), axis=1)

print("3. Yapay Zeka Sınavı Çözüyor (Ensemble Tahminleri)...")
# Eğitimde kullandığımız kolonları aynı sırayla test setinden seçiyoruz
X_test = test_df[features]

# Eğitim aşamasında 5 parçaya bölüp 5 farklı model eğitmiştik.
# Şimdi bu 5 modelin de tahminlerini alıp ortalamasını bularak (Ensemble) daha güçlü bir karar vereceğiz.
test_preds = np.zeros(len(test_df))
for model in trained_models:
    test_preds += model.predict_proba(X_test)[:, 1] / len(trained_models)

print("4. Kaggle Submission Dosyası Oluşturuluyor...")
# Eğitim aşamasında bulduğumuz o 'best_thresh' (optimum eşik) değerini kullanıyoruz
final_predictions = (test_preds > best_thresh).astype(int)

# Sadece Kaggle'ın istediği 2 kolonu (id ve prediction) alarak tabloyu oluşturuyoruz
submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': final_predictions
})

# Dosyayı Google Drive'ına fiziksel bir CSV olarak kaydediyoruz
submission_path = '/content/drive/MyDrive/Trendyol ETicaret/trendyol-e-ticaret-yarismasi-2026-kaggle/my_first_submission.csv'
submission.to_csv(submission_path, index=False)

print("--------------------------------------------------")
print("🎉 BÜYÜK FİNAL BAŞARILI!")
print(f"Kaggle'a yüklenecek dosya hazırlandı: my_first_submission.csv")
print("\nTahmin Dağılımı:")
print(submission['prediction'].value_counts())
print("--------------------------------------------------")

1. Sınav Kağıdı (Test Verisi) Hazırlanıyor...
2. Özellik Mühendisliği Test Setine Uygulanıyor (Bu işlem biraz zaman alacak!)...
   - Matematiksel benzerlikler hesaplanıyor (Lütfen kahvenizi alın, bu işlem 15-20 dakika sürebilir)...
3. Yapay Zeka Sınavı Çözüyor (Ensemble Tahminleri)...
4. Kaggle Submission Dosyası Oluşturuluyor...
--------------------------------------------------
🎉 BÜYÜK FİNAL BAŞARILI!
Kaggle'a yüklenecek dosya hazırlandı: my_first_submission.csv

Tahmin Dağılımı:
prediction
0    1871674
1    1488005
Name: count, dtype: int64
--------------------------------------------------
